# Entrenamiento de IA para Magic Jewelry (NES) con PPO (2M Pasos)

Entrenamiento de una red neuronal profunda mediante **Aprendizaje por Refuerzo (Deep RL)** utilizando el algoritmo **PPO (Proximal Policy Optimization)** sobre el entorno de simulación de *Magic Jewelry*.

### Especificaciones del entorno:
1. **Recompensa por Adyacencia (Migajas de pan):** Premia con 1/4 del valor de una gema por cada color idéntico emparejado en el tablero.
2. **Simulación de la Gema Mágica (Cruces):** Cada 70 gemas destruidas aparece la pieza de cruces, que elimina todo el color sobre el que aterriza.
3. **Entrenamiento extendido a 2.000.000 de pasos:** Permite a la IA dominar la planificación a largo plazo y el uso táctico de la gema mágica.

### Artefactos de salida:
- `modelo_magic_jewelry.onnx`: Red exportada para correr en el navegador web con TypeScript/Phaser.
- `reporte_entrenamiento.csv`: Datos de cada partida jugada.
- `grafica_entrenamiento.png`: Curva de aprendizaje visual con media móvil.

### 1. Instalación de Dependencias

In [ ]:
!pip install --quiet gymnasium stable-baselines3 onnx onnxruntime onnxscript pandas matplotlib

### 2. Definición del Entorno de Simulación con Gema Mágica y Adyacencia

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class EntornoMagicJewelry(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    FILAS = 13
    COLUMNAS = 6
    NUM_COLORES_ESTANDAR = 6
    COLOR_GEMA_MAGICA = 7
    PUNTOS_POR_GEMA = 30
    JOYAS_POR_NIVEL = 70

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(18)
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(84,),
            dtype=np.float32
        )

        self.tablero = np.zeros((self.FILAS, self.COLUMNAS), dtype=np.int8)
        self.pieza_actual = np.zeros(3, dtype=np.int8)
        self.pieza_siguiente = np.zeros(3, dtype=np.int8)

        self.gemas_destruidas = 0
        self.siguiente_umbral_magica = self.JOYAS_POR_NIVEL
        self.proxima_es_magica = False

        self.puntaje_acumulado = 0
        self.piezas_colocadas = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.tablero.fill(0)
        self.puntaje_acumulado = 0
        self.piezas_colocadas = 0
        self.gemas_destruidas = 0
        self.siguiente_umbral_magica = self.JOYAS_POR_NIVEL
        self.proxima_es_magica = False

        self.pieza_actual = self._generar_pieza_aleatoria()
        self.pieza_siguiente = self._generar_pieza_aleatoria()
        return self._obtener_observacion(), {}

    def step(self, action: int):
        columna = action // 3
        rotacion = action % 3

        es_magica = (self.pieza_actual[0] == self.COLOR_GEMA_MAGICA)
        if not es_magica:
            pieza_rotada = self._rotar_pieza(self.pieza_actual, rotacion)
        else:
            pieza_rotada = self.pieza_actual.copy()

        tablero_previo = self.tablero.copy()
        filas_ocupadas, desborde = self._colocar_y_caer(columna, pieza_rotada)
        self.piezas_colocadas += 1

        # Recompensa intermedia (migaja de pan) por conectar colores vecinos iguales
        bonus_adyacencia = 0.0
        if not es_magica and not desborde:
            conexiones_pares = self._contar_conexiones_adyacentes(filas_ocupadas, columna, tablero_previo)
            # Un cuarto del puntaje de una gema (7.5 puntos -> 0.75 en escala de recompensa)
            bonus_adyacencia = conexiones_pares * (self.PUNTOS_POR_GEMA * 0.25 * 0.1)

        if es_magica:
            gemas_eliminadas, puntos_turno = self._procesar_impacto_magica(filas_ocupadas, columna)
        else:
            gemas_eliminadas, puntos_turno = self._resolver_cascadas()

        terminado = False
        if desborde and gemas_eliminadas == 0:
            terminado = True
        elif self.tablero[0, 3] != 0:
            terminado = True

        recompensa = self._calcular_recompensa(puntos_turno, gemas_eliminadas, bonus_adyacencia, terminado)
        self.puntaje_acumulado += puntos_turno
        self.gemas_destruidas += gemas_eliminadas

        # Activar gema mágica cada 70 gemas destruidas
        if self.gemas_destruidas >= self.siguiente_umbral_magica:
            self.proxima_es_magica = True
            while self.siguiente_umbral_magica <= self.gemas_destruidas:
                self.siguiente_umbral_magica += self.JOYAS_POR_NIVEL

        if not terminado:
            self.pieza_actual = self.pieza_siguiente.copy()
            if self.proxima_es_magica:
                self.pieza_siguiente = np.array([self.COLOR_GEMA_MAGICA] * 3, dtype=np.int8)
                self.proxima_es_magica = False
            else:
                self.pieza_siguiente = self._generar_pieza_aleatoria()

        info = {
            "puntaje": self.puntaje_acumulado,
            "piezas": self.piezas_colocadas,
            "gemas_destruidas": self.gemas_destruidas,
            "es_magica": es_magica
        }
        return self._obtener_observacion(), recompensa, terminado, False, info

    def _generar_pieza_aleatoria(self) -> np.ndarray:
        return self.np_random.integers(1, self.NUM_COLORES_ESTANDAR + 1, size=3, dtype=np.int8)

    def _rotar_pieza(self, pieza: np.ndarray, rotacion: int) -> np.ndarray:
        if rotacion == 0:
            return pieza.copy()
        elif rotacion == 1:
            return np.array([pieza[2], pieza[0], pieza[1]], dtype=np.int8)
        else:
            return np.array([pieza[1], pieza[2], pieza[0]], dtype=np.int8)

    def _colocar_y_caer(self, columna: int, pieza: np.ndarray):
        fila_primer_bloque = self.FILAS
        for f in range(self.FILAS):
            if self.tablero[f, columna] != 0:
                fila_primer_bloque = f
                break

        fila_asiento = fila_primer_bloque - 1
        desborde = False
        filas_ocupadas = []

        for i in range(3):
            fila_destino = fila_asiento - i
            gema = pieza[2 - i]
            if fila_destino >= 0:
                self.tablero[fila_destino, columna] = gema
                filas_ocupadas.append(fila_destino)
            else:
                desborde = True
        return filas_ocupadas, desborde

    def _contar_conexiones_adyacentes(self, filas_ocupadas: list, columna: int, tablero_previo: np.ndarray) -> int:
        conexiones = 0
        direcciones = [
            (-1, 0), (1, 0), (0, -1), (0, 1),   # Ortogonales
            (-1, -1), (-1, 1), (1, -1), (1, 1)  # Diagonales
        ]
        for f in filas_ocupadas:
            color = self.tablero[f, columna]
            if color == 0:
                continue
            for df, dc in direcciones:
                nf, nc = f + df, columna + dc
                if 0 <= nf < self.FILAS and 0 <= nc < self.COLUMNAS:
                    if tablero_previo[nf, nc] == color:
                        conexiones += 1
        return conexiones

    def _procesar_impacto_magica(self, filas_ocupadas: list, columna: int):
        if not filas_ocupadas:
            return 0, 0
        fila_asiento = max(filas_ocupadas)
        fila_debajo = fila_asiento + 1
        gemas_eliminadas_color = 0
        if fila_debajo < self.FILAS:
            color_objetivo = self.tablero[fila_debajo, columna]
            if color_objetivo != 0 and color_objetivo != self.COLOR_GEMA_MAGICA:
                mascara_color = (self.tablero == color_objetivo)
                gemas_eliminadas_color = int(np.sum(mascara_color))
                self.tablero[mascara_color] = 0

        self.tablero[self.tablero == self.COLOR_GEMA_MAGICA] = 0
        self._aplicar_gravedad()
        total_destruidas = gemas_eliminadas_color + 3
        puntos_magica = total_destruidas * self.PUNTOS_POR_GEMA
        gemas_cascadas, puntos_cascadas = self._resolver_cascadas()
        return (total_destruidas + gemas_cascadas), (puntos_magica + puntos_cascadas)

    def _buscar_matches(self) -> set:
        gemas_a_eliminar = set()
        for f in range(self.FILAS):
            for c in range(self.COLUMNAS):
                color = self.tablero[f, c]
                if color == 0 or color == self.COLOR_GEMA_MAGICA:
                    continue
                if c + 2 < self.COLUMNAS and color == self.tablero[f, c + 1] == self.tablero[f, c + 2]:
                    gemas_a_eliminar.update({(f, c), (f, c + 1), (f, c + 2)})
                if f + 2 < self.FILAS and color == self.tablero[f + 1, c] == self.tablero[f + 2, c]:
                    gemas_a_eliminar.update({(f, c), (f + 1, c), (f + 2, c)})
                if f + 2 < self.FILAS and c + 2 < self.COLUMNAS and color == self.tablero[f + 1, c + 1] == self.tablero[f + 2, c + 2]:
                    gemas_a_eliminar.update({(f, c), (f + 1, c + 1), (f + 2, c + 2)})
                if f + 2 < self.FILAS and c - 2 >= 0 and color == self.tablero[f + 1, c - 1] == self.tablero[f + 2, c - 2]:
                    gemas_a_eliminar.update({(f, c), (f + 1, c - 1), (f + 2, c - 2)})
        return gemas_a_eliminar

    def _aplicar_gravedad(self):
        for c in range(self.COLUMNAS):
            columna_datos = self.tablero[:, c]
            gemas_vivas = columna_datos[columna_datos != 0]
            cantidad_ceros = self.FILAS - len(gemas_vivas)
            self.tablero[:cantidad_ceros, c] = 0
            self.tablero[cantidad_ceros:, c] = gemas_vivas

    def _resolver_cascadas(self):
        total_gemas_destruidas = 0
        puntos_totales = 0
        paso_combo = 1
        while True:
            matches = self._buscar_matches()
            if not matches:
                break
            cantidad = len(matches)
            total_gemas_destruidas += cantidad
            puntos_totales += cantidad * self.PUNTOS_POR_GEMA * paso_combo
            for f, c in matches:
                self.tablero[f, c] = 0
            self._aplicar_gravedad()
            paso_combo += 1
        return total_gemas_destruidas, puntos_totales

    def _calcular_recompensa(self, puntos_turno: int, gemas_eliminadas: int, bonus_adyacencia: float, terminado: bool) -> float:
        if terminado:
            return -100.0
        recompensa = (puntos_turno * 0.1) + bonus_adyacencia + 1.0
        alturas = [
            self.FILAS - np.argmax(self.tablero[:, c] != 0)
            if np.any(self.tablero[:, c] != 0) else 0
            for c in range(self.COLUMNAS)
        ]
        altura_maxima = max(alturas)
        if altura_maxima > 8:
            recompensa -= (altura_maxima - 8) * 0.5
        return float(recompensa)

    def _obtener_observacion(self) -> np.ndarray:
        tablero_norm = (self.tablero.flatten() / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        pieza_actual_norm = (self.pieza_actual / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        pieza_sig_norm = (self.pieza_siguiente / float(self.COLOR_GEMA_MAGICA)).astype(np.float32)
        return np.concatenate([tablero_norm, pieza_actual_norm, pieza_sig_norm])

### 3. Callback de Métricas en Vivo y Generación de Reportes

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3.common.callbacks import BaseCallback

class CallbackMetricasEntrenamiento(BaseCallback):
    def __init__(self, frecuencia_reporte=100, directorio_salida=".", verbose=1):
        super().__init__(verbose)
        self.frecuencia_reporte = frecuencia_reporte
        self.directorio_salida = directorio_salida
        self.historial_episodios = []

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", [])
        infos = self.locals.get("infos", [])

        for i, terminado in enumerate(dones):
            if terminado:
                info = infos[i]
                registro = {
                    "partida": len(self.historial_episodios) + 1,
                    "paso_global": self.num_timesteps,
                    "puntaje": info.get("puntaje", 0),
                    "piezas_colocadas": info.get("piezas", 0),
                    "gemas_destruidas": info.get("gemas_destruidas", 0)
                }
                self.historial_episodios.append(registro)

                if len(self.historial_episodios) % self.frecuencia_reporte == 0:
                    ultimos = self.historial_episodios[-self.frecuencia_reporte:]
                    media_puntos = np.mean([r["puntaje"] for r in ultimos])
                    media_piezas = np.mean([r["piezas_colocadas"] for r in ultimos])
                    mejor_puntaje = np.max([r["puntaje"] for r in self.historial_episodios])

                    print(
                        f"[Partida {len(self.historial_episodios)}] "
                        f"Paso: {self.num_timesteps} | "
                        f"Media Pts (últimas {self.frecuencia_reporte}): {media_puntos:.1f} | "
                        f"Supervivencia Media: {media_piezas:.1f} piezas | "
                        f"Puntaje Récord: {mejor_puntaje}"
                    )
        return True

    def _on_training_end(self) -> None:
        if not self.historial_episodios:
            return

        df = pd.DataFrame(self.historial_episodios)
        ruta_csv = os.path.join(self.directorio_salida, "reporte_entrenamiento.csv")
        df.to_csv(ruta_csv, index=False)
        print(f"\n[Reporte] Métricas guardadas en '{ruta_csv}'")

        plt.figure(figsize=(12, 9))
        plt.subplot(3, 1, 1)
        plt.plot(df["puntaje"], alpha=0.25, color="#888888", label="Puntaje por partida")
        plt.plot(df["puntaje"].rolling(50, min_periods=1).mean(), color="#0066cc", linewidth=2, label="Media móvil (50 partidas)")
        plt.title("Evolución del Aprendizaje (Magic Jewelry PPO - 2M Pasos)", fontsize=14, fontweight="bold")
        plt.ylabel("Puntos")
        plt.legend(loc="upper left")
        plt.grid(True, alpha=0.3)

        plt.subplot(3, 1, 2)
        plt.plot(df["piezas_colocadas"], alpha=0.25, color="#888888")
        plt.plot(df["piezas_colocadas"].rolling(50, min_periods=1).mean(), color="#009944", linewidth=2, label="Media móvil piezas (supervivencia)")
        plt.ylabel("Piezas colocadas")
        plt.legend(loc="upper left")
        plt.grid(True, alpha=0.3)

        plt.subplot(3, 1, 3)
        plt.plot(df["gemas_destruidas"], alpha=0.25, color="#888888")
        plt.plot(df["gemas_destruidas"].rolling(50, min_periods=1).mean(), color="#9900cc", linewidth=2, label="Media móvil gemas eliminadas")
        plt.xlabel("Número de partida (Episodio)")
        plt.ylabel("Gemas eliminadas")
        plt.legend(loc="upper left")
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        ruta_grafica = os.path.join(self.directorio_salida, "grafica_entrenamiento.png")
        plt.savefig(ruta_grafica, dpi=200)
        plt.close()
        print(f"[Reporte] Gráfica guardada en '{ruta_grafica}'")

### 4. Configuración del Modelo PPO con 8 Entornos en Paralelo

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

NUM_ENTORNOS = 8
env_vectorizado = make_vec_env(EntornoMagicJewelry, n_envs=NUM_ENTORNOS)

policy_kwargs = dict(
    net_arch=dict(pi=[128, 128], vf=[128, 128])
)

modelo = PPO(
    policy="MlpPolicy",
    env=env_vectorizado,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=128,
    n_epochs=10,
    gamma=0.99,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs,
    verbose=0
)

print("Modelo PPO configurado exitosamente.")

### 5. Entrenamiento de 2.000.000 de Pasos con Monitoreo en Vivo

In [ ]:
callback_metricas = CallbackMetricasEntrenamiento(frecuencia_reporte=100)

PASOS_TOTALES = 2_000_000
print(f"--- INICIANDO ENTRENAMIENTO EXTENDIDO ({PASOS_TOTALES:,} PASOS) ---\n")

modelo.learn(total_timesteps=PASOS_TOTALES, callback=callback_metricas)

modelo.save("magic_jewelry_ppo_2m_final")
print("\n--- ENTRENAMIENTO FINALIZADO CON ÉXITO ---")

### 6. Exportación a Formato ONNX (Compatible con CPU y Navegador Web)

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

class PoliticaActorONNX(nn.Module):
    def __init__(self, policy):
        super().__init__()
        # Extraer extractor y red de acciones forzando CPU
        self.mlp_extractor = policy.mlp_extractor.policy_net.to('cpu')
        self.action_net = policy.action_net.to('cpu')

    def forward(self, x):
        caracteristicas = self.mlp_extractor(x)
        logits_acciones = self.action_net(caracteristicas)
        return torch.argmax(logits_acciones, dim=-1)

# 1. Mover todo el modelo entrenado a CPU para evitar conflictos de CUDA al exportar
modelo.policy.to('cpu')
red_actor = PoliticaActorONNX(modelo.policy)
red_actor.eval()

# 2. Tensor de prueba explícitamente en CPU
tensor_ejemplo = torch.zeros((1, 84), dtype=torch.float32, device='cpu')
NOMBRE_ARCHIVO_ONNX = "modelo_magic_jewelry.onnx"

# 3. Exportar a ONNX
torch.onnx.export(
    red_actor,
    tensor_ejemplo,
    NOMBRE_ARCHIVO_ONNX,
    input_names=["observacion"],
    output_names=["accion"],
    opset_version=18
)

# 4. Comprobación y unificación
modelo_onnx = onnx.load(NOMBRE_ARCHIVO_ONNX)
onnx.checker.check_model(modelo_onnx)

# 5. Inferencia de prueba con ONNX Runtime
sesion_ort = ort.InferenceSession(NOMBRE_ARCHIVO_ONNX)
salida_prueba = sesion_ort.run(None, {"observacion": np.zeros((1, 84), dtype=np.float32)})

print(f"\n¡ARCHIVO ONNX GENERADO CON ÉXITO!: '{NOMBRE_ARCHIVO_ONNX}'")
print(f"Inferencia de prueba: Acción elegida = {salida_prueba[0][0]}")
print("\nArtefactos de salida exportados:")
print("1. modelo_magic_jewelry.onnx")
print("2. reporte_entrenamiento.csv")
print("3. grafica_entrenamiento.png")